### Purpose

This notebook reproduces the Human and GPT-4 exposure measures using the Eloundou et al. α, β, and γ scoring definitions and aggregates the task-level exposure labels to the occupation level using the prescribed task weights.

The resulting exposure benchmark is kept as an independent RQ1 construct and is later integrated with the occupational task-affordance configuration.

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes"
)

USABLE_FILE =  (
    BASE_DIR
    / "06_Eloundou_Task_Alignment"
    / "Output"
    / "Occupational_Task_Eloundou_Exposure_Usable_v1.csv"
)

tasks = pd.read_csv(
    USABLE_FILE,
    dtype={"soc6": str}
)

print("Rows:", len(tasks))
print("Unique task IDs:", tasks["task_id_match"].nunique())
print("Unique SOC6:", tasks["soc6"].nunique())

print("\nMissing required fields:")
print(
    tasks[
        [
            "task_id_match",
            "soc6",
            "human_exposure_agg",
            "gpt4_exposure"
        ]
    ].isna().sum()
)

print("\nHuman exposure labels:")
print(tasks["human_exposure_agg"].value_counts(dropna=False))

print("\nGPT-4 exposure labels:")
print(tasks["gpt4_exposure"].value_counts(dropna=False))

In [ ]:
# ------------------------------------------------------------------
# Step 2 — Create task-level Human and GPT-4 exposure scores
# ------------------------------------------------------------------

score_map_alpha = {"E0": 0.0, "E1": 1.0, "E2": 0.0}
score_map_beta  = {"E0": 0.0, "E1": 1.0, "E2": 0.5}
score_map_gamma = {"E0": 0.0, "E1": 1.0, "E2": 1.0}

for source, column in [
    ("human", "human_exposure_agg"),
    ("gpt4", "gpt4_exposure"),
]:
    tasks[f"{source}_alpha"] = tasks[column].map(score_map_alpha)
    tasks[f"{source}_beta"] = tasks[column].map(score_map_beta)
    tasks[f"{source}_gamma"] = tasks[column].map(score_map_gamma)

score_cols = [
    "human_alpha",
    "human_beta",
    "human_gamma",
    "gpt4_alpha",
    "gpt4_beta",
    "gpt4_gamma",
]

assert tasks[score_cols].notna().all().all()

print("Task-level Eloundou scores created for:", len(tasks))

In [ ]:
# ------------------------------------------------------------------
# Step 3 — Create strict Eloundou task weights
# ------------------------------------------------------------------

tasks["eloundou_task_weight"] = (
    tasks["eloundou_task_type"]
    .map({
        "Core": 2,
        "Supplemental": 1,
    })
    .fillna(1)
    .astype(int)
)

assert tasks["eloundou_task_weight"].notna().all()

print(
    tasks["eloundou_task_weight"]
    .value_counts()
    .sort_index()
)

print(
    "Total Eloundou task weight:",
    tasks["eloundou_task_weight"].sum(),
)

In [ ]:
# ------------------------------------------------------------------
# Step 4 — Aggregate Eloundou exposure to SOC6
# ------------------------------------------------------------------

measures = [
    "human_alpha",
    "human_beta",
    "human_gamma",
    "gpt4_alpha",
    "gpt4_beta",
    "gpt4_gamma",
]

for measure in measures:
    tasks[f"{measure}_weighted"] = (
        tasks[measure]
        * tasks["eloundou_task_weight"]
    )

occupation_exposure = (
    tasks.groupby("soc6", as_index=False)
    .agg(
        matched_task_count=("task_id_match", "nunique"),
        total_task_weight=("eloundou_task_weight", "sum"),
        human_alpha_sum=("human_alpha_weighted", "sum"),
        human_beta_sum=("human_beta_weighted", "sum"),
        human_gamma_sum=("human_gamma_weighted", "sum"),
        gpt4_alpha_sum=("gpt4_alpha_weighted", "sum"),
        gpt4_beta_sum=("gpt4_beta_weighted", "sum"),
        gpt4_gamma_sum=("gpt4_gamma_weighted", "sum"),
    )
)

for measure in measures:
    occupation_exposure[measure] = (
        occupation_exposure[f"{measure}_sum"]
        / occupation_exposure["total_task_weight"]
    )

assert len(occupation_exposure) == 604
assert occupation_exposure["soc6"].is_unique
assert occupation_exposure[measures].notna().all().all()

print(f"Occupation benchmark records: {len(occupation_exposure):,}")
print(f"Unique SOC6: {occupation_exposure['soc6'].nunique():,}")

In [ ]:
# ------------------------------------------------------------------
# Step 5 — Validate Eloundou benchmark measures
# ------------------------------------------------------------------

import numpy as np

for source in ["human", "gpt4"]:
    alpha = occupation_exposure[f"{source}_alpha"]
    beta = occupation_exposure[f"{source}_beta"]
    gamma = occupation_exposure[f"{source}_gamma"]

    assert alpha.between(0, 1).all()
    assert beta.between(0, 1).all()
    assert gamma.between(0, 1).all()

    assert (alpha <= beta).all()
    assert (beta <= gamma).all()

    assert np.allclose(
        beta,
        (alpha + gamma) / 2,
        atol=1e-12,
    )

print("All Eloundou scoring identities validated.")

display(
    occupation_exposure[measures]
    .describe()
    .T[
        ["count", "mean", "std", "min", "max"]
    ]
    .round(4)
)

In [ ]:
# ------------------------------------------------------------------
# Step 5 — Create the final occupation-level benchmark table
# ------------------------------------------------------------------

eloundou_exposure_benchmark = occupation_exposure[
    [
        "soc6",
        "matched_task_count",
        "total_task_weight",
        "human_alpha",
        "human_beta",
        "human_gamma",
        "gpt4_alpha",
        "gpt4_beta",
        "gpt4_gamma",
    ]
].copy()

assert len(eloundou_exposure_benchmark) == 604
assert eloundou_exposure_benchmark["soc6"].nunique() == 604
assert eloundou_exposure_benchmark[measures].notna().all().all()

print("Final benchmark records:", len(eloundou_exposure_benchmark))
display(eloundou_exposure_benchmark.head(10))

In [ ]:
# ------------------------------------------------------------------
# Step 6 — Save combined Eloundou exposure benchmark
# ------------------------------------------------------------------

OUTPUT_FILE = (
    BASE_DIR
    / "06_Eloundou_Task_Alignment"
    / "Output"
    / "Occupational_Eloundou_Exposure_Benchmark_v1.csv"
)

eloundou_exposure_benchmark.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:")
print(OUTPUT_FILE)